# OsteoLens Phase 4 — checkpoint and Grad-CAM verification

This notebook runs the **existing** FastAPI handler and Grad-CAM implementation against the trained checkpoint and five real BTXRD sample images. It does not train or alter model weights. The checkpoint remains in Google Drive; the small companion bundle contains only the application code, manifest, and 30 already-curated sample images.

A separate optional section audits LabelMe annotation-to-image matching if the full BTXRD archive is available in Colab. It does not train or claim a localization model.


## Before you run

1. Open this notebook in Google Colab and select a GPU runtime if available.
2. The checkpoint is searched for under `MyDrive/OsteoLensRuns/BTXRD/osteolens_btxrd_resnet18.pth`; if it is elsewhere in MyDrive, the notebook will search for it.
3. Run the cells in order. Upload `phase4_verification_bundle.zip` when prompted. The bundle is only a few MB and does **not** contain the full BTXRD dataset or checkpoint.
4. The notebook checks Colab's existing PyTorch/torchvision first and does not replace them. The checkpoint SHA-256 is pinned to `32ee1c1c2b320865ade06ebdf819c2f6b3cb29e8c695ce9ea3124ecfae440756`; a different file is rejected. Its training metadata contains PyTorch's `TorchVersion` string subclass. The backend allowlists only that type inside a scoped `weights_only=True` load; it never uses unrestricted pickle loading. The checkpoint records PyTorch `2.14.1+cu130` and torchvision `0.29.1+cu130`; this is the recommended exact pair if the Colab runtime's pair is incompatible. The official PyTorch install matrix lists matching 2.14.1/0.29.1 wheels. Do not install a new torch build unless the runtime fails compatibility checks.
5. The model section tests five different source images (Normal, Benign, Malignant plus two more), prints model-generated outputs, verifies unique Grad-CAMs and overlay dimensions, checks invalid input errors, and saves a report and visualizations to `MyDrive/OsteoLensRuns/Phase4Verification/`.
6. The final cell optionally audits LabelMe JSON files if `/content/BTXRD` or an extracted BTXRD folder on Drive is available. If the full dataset is not present in this Colab runtime, that audit is reported as skipped.

This notebook is for research verification. Grad-CAM is qualitative attribution and is not a tumor boundary or diagnosis.


In [ ]:
# Install only missing non-PyTorch test dependencies. Never replace Colab's torch/torchvision here.
import importlib.util, subprocess, sys

required = {'fastapi': 'fastapi>=0.115,<1', 'httpx': 'httpx>=0.27,<1',
            'PIL': 'Pillow>=10.4,<13', 'numpy': 'numpy>=1.26,<3',
            'matplotlib': 'matplotlib>=3.9,<4'}
missing = [spec for module, spec in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing non-PyTorch packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
else:
    print('Verification dependencies are already available.')

TORCH_RUNTIME_ERROR = None
try:
    import torch, torchvision
    from torchvision.models import resnet18
    resnet18(weights=None)  # check the installed torch/torchvision pair can construct ResNet-18
except Exception as exc:
    TORCH_RUNTIME_ERROR = f'{type(exc).__name__}: {exc}'
print('Python:', sys.version.split()[0])
if TORCH_RUNTIME_ERROR:
    print('FAIL torch/torchvision compatibility:', TORCH_RUNTIME_ERROR)
    print('Checkpoint runtime recorded torch 2.14.1+cu130 and torchvision 0.29.1+cu130. See the guide for a matching pair; no packages were replaced.')
else:
    print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Mount Drive, locate the existing checkpoint, and upload the small source bundle.
from pathlib import Path
from google.colab import drive, files
import hashlib, os, shutil, zipfile

drive.mount('/content/drive')
MY_DRIVE = Path('/content/drive/MyDrive')
expected_checkpoint = MY_DRIVE / 'OsteoLensRuns' / 'BTXRD' / 'osteolens_btxrd_resnet18.pth'
CHECKPOINT_DRIVE = expected_checkpoint if expected_checkpoint.is_file() else None
if CHECKPOINT_DRIVE is None:
    candidates = sorted(MY_DRIVE.rglob('osteolens_btxrd_resnet18.pth'))
    if len(candidates) == 1: CHECKPOINT_DRIVE = candidates[0]
if CHECKPOINT_DRIVE is None:
    CHECKPOINT_SEARCH_ERROR = 'Checkpoint file was not found in MyDrive.'
    print('FAIL:', CHECKPOINT_SEARCH_ERROR)
else:
    CHECKPOINT_SEARCH_ERROR = None
    print('Checkpoint in Drive:', CHECKPOINT_DRIVE)
    print('Checkpoint bytes:', CHECKPOINT_DRIVE.stat().st_size)

BUNDLE_PATH = Path('/content/phase4_verification_bundle.zip')
if not BUNDLE_PATH.is_file():
    uploaded = files.upload()
    uploaded_zip = next((name for name in uploaded if name.lower().endswith('.zip')), None)
    if uploaded_zip is None: raise FileNotFoundError('Upload phase4_verification_bundle.zip.')
    shutil.move(uploaded_zip, BUNDLE_PATH)
BUNDLE_SHA256 = 'c73fe6ecbba349a5bd5fa5a26758807f668b885c6d5b1417862ad93496237d73'
actual_bundle_sha256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
if actual_bundle_sha256 != BUNDLE_SHA256:
    raise ValueError(f'Verification bundle checksum mismatch: {actual_bundle_sha256}')
print('Verification bundle SHA-256 passed:', actual_bundle_sha256)

PROJECT_ROOT = Path('/content/osteolens_phase4_verify/project')
if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    bad_member = archive.testzip()
    if bad_member: raise ValueError(f'Bundle ZIP CRC failed for {bad_member}')
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        if PROJECT_ROOT.resolve() not in target.parents and target != PROJECT_ROOT.resolve():
            raise ValueError(f'Unsafe path in verification bundle: {member.filename}')
    archive.extractall(PROJECT_ROOT)

EXPECTED_CHECKPOINT_SHA256 = '32ee1c1c2b320865ade06ebdf819c2f6b3cb29e8c695ce9ea3124ecfae440756'
checkpoint_local = None
CHECKPOINT_SHA256 = None
if CHECKPOINT_DRIVE is not None:
    checkpoint_local = PROJECT_ROOT / 'models' / 'checkpoints' / 'osteolens_btxrd_resnet18.pth'
    checkpoint_local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CHECKPOINT_DRIVE, checkpoint_local)
    if checkpoint_local.stat().st_size != CHECKPOINT_DRIVE.stat().st_size or checkpoint_local.stat().st_size == 0:
        CHECKPOINT_SEARCH_ERROR = 'Checkpoint copy is incomplete.'
        checkpoint_local.unlink(missing_ok=True)
        checkpoint_local = None
    else:
        CHECKPOINT_SHA256 = hashlib.sha256(checkpoint_local.read_bytes()).hexdigest()
        if CHECKPOINT_SHA256 != EXPECTED_CHECKPOINT_SHA256:
            checkpoint_local.unlink(missing_ok=True)
            raise ValueError(f'Checkpoint SHA-256 mismatch; expected {EXPECTED_CHECKPOINT_SHA256}, got {CHECKPOINT_SHA256}. Refusing to load.')
        print('Checkpoint copied and SHA-256 verified:', CHECKPOINT_SHA256)
else:
    CHECKPOINT_SHA256 = None


In [ ]:
# Import the unchanged API and let its own lazy loader attempt the checkpoint load.
import sys
sys.path.insert(0, str(PROJECT_ROOT))
if checkpoint_local is not None:
    os.environ['OSTEOLENS_MODEL_PATH'] = str(checkpoint_local)

import backend.main as backend_module
from fastapi.testclient import TestClient

MODEL_LOAD_ERROR = TORCH_RUNTIME_ERROR or CHECKPOINT_SEARCH_ERROR
model_bundle = None
if MODEL_LOAD_ERROR is None:
    try:
        model_bundle = backend_module._get_model()
        assert model_bundle.class_names == ('Normal', 'Benign', 'Malignant'), model_bundle.class_names
        assert model_bundle.input_size == 256, model_bundle.input_size
        assert model_bundle.model_version == 'btxrd-resnet18-seed42', model_bundle.model_version
        assert model_bundle.model.fc.out_features == 3
        print('PASS checkpoint loaded by backend:', model_bundle.model_version)
        print('Class mapping:', model_bundle.class_names, '| input size:', model_bundle.input_size, '| device:', model_bundle.device)
    except Exception as exc:
        MODEL_LOAD_ERROR = f'{type(exc).__name__}: {exc}'
        print('FAIL backend checkpoint load:', MODEL_LOAD_ERROR)
else:
    print('SKIP backend model load:', MODEL_LOAD_ERROR)
client = TestClient(backend_module.app)
health = client.get('/api/health')
print('Backend health response:', health.status_code, health.json())


In [ ]:
# Call the real FastAPI endpoint on five different, checksum-verified BTXRD cases.
import base64, io, json
from collections import Counter
from PIL import Image, ImageChops, ImageOps
import numpy as np

manifest = json.loads((PROJECT_ROOT / 'src' / 'data' / 'cases.json').read_text(encoding='utf-8'))
records = manifest['cases']
chosen = []
for source_label in ('Normal', 'Benign', 'Malignant'):
    chosen.append(next(record for record in records if record['classification'] == source_label))
for record in records:
    if record['case_id'] not in {item['case_id'] for item in chosen} and len(chosen) < 5:
        chosen.append(record)
if len({item['case_id'] for item in chosen}) != 5:
    raise RuntimeError('Could not choose five distinct manifest cases.')

RESULTS_DIR = MY_DRIVE / 'OsteoLensRuns' / 'Phase4Verification'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
verification_rows = []
failures = []
heatmap_hashes = []

def decode_png_data_url(value):
    prefix = 'data:image/png;base64,'
    if not isinstance(value, str) or not value.startswith(prefix):
        raise AssertionError('Backend did not return a PNG data URL.')
    raw = base64.b64decode(value[len(prefix):], validate=True)
    image = Image.open(io.BytesIO(raw)).convert('RGB')
    return raw, image

for record in chosen:
    image_path = PROJECT_ROOT / 'public' / 'images' / 'btxrd' / record['image_id']
    image_bytes = image_path.read_bytes()
    actual_sha = hashlib.sha256(image_bytes).hexdigest()
    if actual_sha != record['sha256']:
        failures.append(f"{record['case_id']}: source image SHA-256 differs from manifest")
        continue
    response = client.post(f"/api/analyze/{record['case_id']}")
    if response.status_code != 200:
        failures.append(f"{record['case_id']}: API HTTP {response.status_code}: {response.text[:500]}")
        continue
    result = response.json()
    try:
        assert result['status'] == 'complete'
        assert result['case_id'] == record['case_id'] and result['image_id'] == record['image_id']
        assert result['source_sha256'] == record['sha256']
        assert result['prediction'] in ('Normal','Benign','Malignant')
        assert result['model_version'] == 'btxrd-resnet18-seed42'
        assert set(result['probabilities']) == {'Normal','Benign','Malignant'}
        assert abs(sum(result['probabilities'].values()) - 1.0) < 1e-4
        heat_bytes, heat = decode_png_data_url(result['gradcam_heatmap_data_url'])
        _, overlay = decode_png_data_url(result['gradcam_overlay_data_url'])
        with Image.open(image_path) as source: original = ImageOps.exif_transpose(source).convert('RGB')
        aligned = heat.size == original.size == overlay.size
        assert aligned, f'Original/heatmap/overlay dimensions differ: {original.size}, {heat.size}, {overlay.size}'
        overlay_change = float(np.asarray(ImageChops.difference(original, overlay), dtype=np.float32).mean())
        assert overlay_change > 0.0, 'Generated overlay is pixel-identical to original.'
        heat_sha = hashlib.sha256(heat_bytes).hexdigest(); heatmap_hashes.append(heat_sha)
        heat.save(RESULTS_DIR / f"{record['case_id']}_gradcam.png")
        overlay.save(RESULTS_DIR / f"{record['case_id']}_overlay.png")
        verification_rows.append({
            'case_id':record['case_id'],'image_id':record['image_id'],
            'source_label':record['classification'],'prediction':result['prediction'],
            'uncalibrated_score':result['confidence'],'probabilities':result['probabilities'],
            'model_version':result['model_version'],'source_sha256_verified':True,
            'heatmap_sha256':heat_sha,'original_size':list(original.size),
            'heatmap_size':list(heat.size),'overlay_size':list(overlay.size),
            'overlay_mean_abs_change':overlay_change,'alignment_check':aligned,
        })
        print(f"PASS {record['case_id']} ({record['classification']}) -> {result['prediction']} "
              f"score={result['confidence']:.4f} size={original.size} heatmap_sha256={heat_sha[:16]}…")
    except Exception as exc:
        failures.append(f"{record['case_id']}: {type(exc).__name__}: {exc}")

heatmaps_distinct = len(heatmap_hashes) == 5 and len(set(heatmap_hashes)) == 5
if not heatmaps_distinct:
    failures.append(f'Grad-CAM uniqueness failed: {len(set(heatmap_hashes))} unique maps for {len(heatmap_hashes)} successful images.')

if model_bundle is None:
    invalid_image_status = 'skipped: model checkpoint did not load'
else:
    from training.src.inference import UnsupportedImageError, generate_gradcam_image_bytes
    try:
        generate_gradcam_image_bytes(b'not a real image', model_bundle)
        invalid_image_status = 'failed: invalid image was accepted'
    except UnsupportedImageError:
        invalid_image_status = 'passed'
    except Exception as exc:
        invalid_image_status = f'failed: unexpected {type(exc).__name__}: {exc}'
if invalid_image_status != 'passed': failures.append(invalid_image_status)

unknown_case_status = 'passed' if client.post('/api/analyze/NOT-A-REAL-CASE').status_code == 404 else 'failed'
if unknown_case_status != 'passed': failures.append('Unknown case ID was not rejected with HTTP 404.')

checks = {
    'checkpoint_loaded_by_backend': model_bundle is not None,
    'five_real_case_api_calls': len(verification_rows) == 5,
    'source_labels_cover_all_three_classes': {row['source_label'] for row in verification_rows} >= {'Normal','Benign','Malignant'},
    'five_image_specific_gradcams': heatmaps_distinct,
    'overlay_alignment_checked_for_each_case': len(verification_rows)==5 and all(row['alignment_check'] for row in verification_rows),
    'invalid_image_rejected': invalid_image_status == 'passed',
    'unknown_case_rejected': unknown_case_status == 'passed',
}
passed = [name for name,value in checks.items() if value]
failed = [name for name,value in checks.items() if not value]
skipped = [] if model_bundle is not None else ['actual model inference and invalid-image model path; checkpoint unavailable or failed to load']
report = {
    'verification':'passed' if not failed and len(verification_rows)==5 else 'failed',
    'passed':passed,'failed':failed,'skipped':skipped,
    'checkpoint_path_in_drive':str(CHECKPOINT_DRIVE) if CHECKPOINT_DRIVE else None,
    'checkpoint_sha256':CHECKPOINT_SHA256,'expected_checkpoint_sha256':EXPECTED_CHECKPOINT_SHA256,'model_version':model_bundle.model_version if model_bundle else None,
    'class_names':list(model_bundle.class_names) if model_bundle else [],
    'pytorch_version':str(torch.__version__) if 'torch' in globals() else None,
    'torchvision_version':str(torchvision.__version__) if 'torchvision' in globals() else None,
    'device':str(model_bundle.device) if model_bundle else None,
    'cases_tested':verification_rows,'source_label_counts':dict(Counter(row['source_label'] for row in verification_rows)),
    'distinct_gradcam_count':len(set(heatmap_hashes)),'invalid_image_test':invalid_image_status,
    'unknown_case_test':unknown_case_status,'failures':failures,
    'localization':'not tested; no localization model is implemented or claimed',
}
(RESULTS_DIR / 'phase4_model_verification.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Verification status:',report['verification'],'| passed:',len(passed),'failed:',len(failed),'skipped:',len(skipped))
print('Saved verification report to:',RESULTS_DIR/'phase4_model_verification.json')


## Optional BTXRD LabelMe annotation matching audit

BTXRD's `.zip` contains its `images/`, `Annotations/` and `dataset.csv`. The project repository intentionally contains only the separate 30-case browser sample, not that 840 MB archive. If `/content/BTXRD` is not available in this runtime, open the existing `training/osteolens_colab.ipynb` in this same Colab runtime and run its setup, dependencies, and verified direct-download/extraction cells (cells 1, 3 and 4). That downloader verifies the official Figshare metadata, size, MD5 and ZIP CRC before extraction. Return here and run the audit cell. The archive stays in Colab temporary storage; this audit writes only a compact JSON report to Drive.

The audit matches LabelMe JSON files to image filenames and `imagePath` values, counts ambiguous/orphan/malformed records and shape types, checks image dimensions, and writes exact per-file matching results. It does not assume every image should have an annotation, assign labels, train a detector, or infer tumor boundaries. Localization should not be proposed until this report is reviewed.


In [ ]:
# Optional: audit the extracted archive's LabelMe JSON joins and source labels.
from training.src.labelme_audit import audit_labelme_dataset

candidate_roots = [Path('/content/BTXRD'), MY_DRIVE / 'datasets' / 'BTXRD', MY_DRIVE / 'BTXRD']
def has_full_btxrd(root):
    if not root.exists(): return False
    return any(p.is_dir() and p.name.casefold()=='images' for p in root.rglob('*')) and any(
        p.is_dir() and p.name.casefold()=='annotations' for p in root.rglob('*'))
dataset_root = next((root for root in candidate_roots if has_full_btxrd(root)), None)
if dataset_root is None:
    annotation_audit = {
        'status': 'skipped',
        'reason': 'Full BTXRD extraction with images/ and Annotations/ is unavailable in this Colab runtime.',
        'how_to_make_available': 'Run setup/dependency/download/extraction cells 1, 3, and 4 in training/osteolens_colab.ipynb in this Colab runtime, then rerun this cell.',
    }
    print('SKIPPED: no full BTXRD annotation archive/extraction found. The 30-case sample is not used to infer annotation coverage.')
else:
    try:
        annotation_audit = audit_labelme_dataset(dataset_root)
        print('Annotation matching summary:', json.dumps({k:v for k,v in annotation_audit.items() if k!='records'},indent=2)[:10000])
    except Exception as exc:
        annotation_audit = {'status':'failed','dataset_root':str(dataset_root),'error':f'{type(exc).__name__}: {exc}'}
        print('FAIL annotation audit:', annotation_audit['error'])

annotation_path = RESULTS_DIR / 'labelme_annotation_matching_audit.json'
annotation_path.write_text(json.dumps(annotation_audit,indent=2),encoding='utf-8')
print('Annotation audit status:', annotation_audit['status'], '| saved:', annotation_path)


In [ ]:
# Print the combined passed/failed/skipped status. The saved model JSON is the run evidence.
print('MODEL VERIFICATION:', report['verification'].upper())
print('PASSED:', report['passed'] or 'none')
print('FAILED:', report['failed'] or 'none')
print('SKIPPED:', report['skipped'] or 'none')
annotation_status = annotation_audit.get('status','skipped')
print(('PASSED' if annotation_status=='completed' else 'SKIPPED' if annotation_status=='skipped' else 'FAILED'), 'full BTXRD annotation join audit:', annotation_status)
print('Model report:', RESULTS_DIR/'phase4_model_verification.json')
print('Annotation audit:', RESULTS_DIR/'labelme_annotation_matching_audit.json')
if report['failed'] or report['verification'] != 'passed':
    print('Phase 4 model integration is NOT verified. Review failures above; do not proceed to Phase 5.')
